In [37]:
from pathlib import Path
import re
import sys

In [38]:
def classify_line(line : str) -> str:
    """Classify an extracted PDF line by its visible structure.
    Args:
        line: Text line read from a PDF page.
    Returns:
        A label such as heading, table-row, or paragraph.
    """

    words = line.split()
    if len(words) <= 10 and (line.isupper() or line.endswith(":")):
        return "heading"
    
    # r"\s{2,}|\|" means:
    #   \s{2,}  -> find 2 or more whitespace characters (spaces/tabs/newlines)
    #   |       -> OR
    #   \|      -> find a literal pipe character |
    
    if len(re.findall(r"\s{2,}|\|")) >= 2:
        return "table-row"

    return "paragraph"


In [39]:
import pymupdf

def parse_pdf(path:Path) -> list[dict]:
    """Extract structured text blocks from a PDF.

    Args:
        path: PDF file to read.

    Returns:
        Records containing each block's text and location.
    """

    try:
        import pymupdf
    except ImportError as e:
        raise RuntimeError("Install pymupdf as PDF support by -> 'pip install pymupdf' ")

    records = []

    with pymupdf.open(path) as pdf:

        for page_number , page in enumerate(pdf , start=1):
            blocks = page.get_text("blocks" , sort = True)

            for block_number , block in enumerate(blocks , start = 1):
                # block is a tuple like -> (x0 , y0 , x1 , y1 , text , ...) -> 4th index contain text
                raw_text = block[4].strip() 

                if raw_text.strip():
                    raw_lines = [line for line in raw_text.splitlines() if line.strip()]

                    kind = (
                        "table-row"
                        if any(classify_line(line) == "table-row" for line in raw_lines)
                        else classify_line(raw_lines[0])
                    )

                    records.append(
                        {
                            "source":path.name,
                            "page":page_number,
                            "block":block_number,
                            "kind":kind,
                            "text":" ".join(raw_text.split()),
                        }
                    )
    return records

In [40]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0 , str(ROOT))


PDF_DIR = ROOT / "data" / "input" / "pdfs"


def main() -> None:
    pdfs = sorted(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"No PDFs Found in {PDF_DIR}. Add a PDF and Re-Run")
        return

    for pdf in pdfs:
        print(f"\n[Load]  ->  {pdf.name}")

        try:
            records = parse_pdf(pdf)
        except RuntimeError as error:
            print(f"[Dependency] {error}")
            return

        pages = {item["page"] for item in records}

In [41]:
main()


[Load]  ->  novasecure_quickstart.pdf


TypeError: findall() missing 1 required positional argument: 'string'